# Pipeline 5 — Graph-Enhanced Hybrid RAG (BM25 + Dense + Entity Graph + 3-Way RRF)

**Method:** Three-signal retrieval via 3-way weighted Reciprocal Rank Fusion.

```
              ┌─── BM25 (full rank) ──────────────┐
Question ─────┼─── Dense (full rank) ─────────────┼──→ 3-Way RRF ──→ top-k ──→ LLM
              └─── Graph PPR (entity graph) ──────┘
```

$$\text{score}(d) = \frac{1}{k + r_{\text{BM25}}} + \frac{1}{k + r_{\text{Dense}}} + 1.5 \cdot \frac{1}{k + r_{\text{Graph}}}$$

**Graph component:** Builds a per-query entity graph (NetworkX), detects cross-paragraph
title mentions (bridge signal), and runs Personalized PageRank to propagate scores.

**Features:** 14-key API carousel, auto workers = key count, spaCy NER

## Step 1 — Install Dependencies

In [ ]:
!pip install -q datasets numpy rank-bm25 sentence-transformers lancedb groq tqdm networkx spacy ragas langchain-groq langchain-core
!python -m spacy download en_core_web_sm -q

## Step 1b — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Path to your project folder on Google Drive ---
BASE_DIR     = '/content/drive/MyDrive/HotPotQA-Coding-Trials'
API_KEYS_CSV = f'{BASE_DIR}/api_keys.csv'
RESULTS_DIR  = f'{BASE_DIR}/results/e5_trials/lancedb'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Drive mounted. Project dir: {BASE_DIR}")

## Step 2 — Imports and Configuration

In [ ]:
import os, json, re, time, random, collections, string, csv, threading
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import lancedb
import networkx as nx
import spacy
from groq import Groq
from datetime import datetime

GROQ_MODEL    = "llama-3.3-70b-versatile"
DENSE_MODEL   = "intfloat/e5-base-v2"
TOP_K         = 4
RRF_K         = 60
GRAPH_WEIGHT  = 1.5
N_SAMPLES     = 100
SEED          = 42
MAX_WORKERS   = None  # auto-set to len(key_manager.keys) after Step 3
PIPELINE_NAME = "GraphRAG_e5-base-v2_LanceDB"
# API_KEYS_CSV and RESULTS_DIR are set in the Drive mount cell above

print(f"Pipeline : {PIPELINE_NAME}")
print(f"LLM      : {GROQ_MODEL} | Dense: {DENSE_MODEL} | Sparse: BM25 | Graph: NER+PPR")
print(f"RRF k={RRF_K}, graph_weight={GRAPH_WEIGHT}, Workers: auto (set after API keys load)")

## Step 3 — API Key Manager (14-Key Carousel)

In [ ]:
class APIKeyManager:
    BUFFERS = {'rpm': 25, 'rpd': 900, 'tpm': 10_000, 'tpd': 90_000}

    def __init__(self, csv_path):
        self.keys = self._load_keys(csv_path)
        self._lock = threading.Lock()
        self._idx = 0
        self._usage = {k: {'min_req': [], 'day_req': [], 'min_tok': [], 'day_tok': []} for k in self.keys}
        print(f"Loaded {len(self.keys)} API keys")

    @staticmethod
    def _load_keys(csv_path):
        keys = []
        with open(csv_path, 'r') as f:
            for row in csv.DictReader(f):
                k = row.get('API_KEY', '').strip()
                if k: keys.append(k)
        if not keys: raise ValueError(f"No keys in {csv_path}")
        return keys

    def _clean(self, key):
        now = time.time()
        u = self._usage[key]
        u['min_req'] = [t for t in u['min_req'] if now - t < 60]
        u['day_req'] = [t for t in u['day_req'] if now - t < 86400]
        u['min_tok'] = [(t, n) for t, n in u['min_tok'] if now - t < 60]
        u['day_tok'] = [(t, n) for t, n in u['day_tok'] if now - t < 86400]

    def _is_available(self, key):
        self._clean(key)
        u = self._usage[key]
        return (len(u['min_req']) < self.BUFFERS['rpm']
                and len(u['day_req']) < self.BUFFERS['rpd']
                and sum(n for _, n in u['min_tok']) < self.BUFFERS['tpm']
                and sum(n for _, n in u['day_tok']) < self.BUFFERS['tpd'])

    def get_key(self):
        with self._lock:
            for _ in range(len(self.keys)):
                key = self.keys[self._idx]
                self._idx = (self._idx + 1) % len(self.keys)
                if self._is_available(key): return key
            return self._wait_and_get()

    def _wait_and_get(self):
        min_wait = 60
        for key in self.keys:
            reqs = self._usage[key]['min_req']
            if reqs: min_wait = min(min_wait, max(0, 60 - (time.time() - min(reqs))))
        print(f"  ⏳ All keys busy — waiting {min_wait:.1f}s...")
        time.sleep(min_wait + 1)
        for _ in range(len(self.keys)):
            key = self.keys[self._idx]
            if self._is_available(key): return key
            self._idx = (self._idx + 1) % len(self.keys)
        return self.keys[self._idx]

    def record(self, key, tokens=0):
        with self._lock:
            now = time.time()
            self._usage[key]['min_req'].append(now)
            self._usage[key]['day_req'].append(now)
            if tokens > 0:
                self._usage[key]['min_tok'].append((now, tokens))
                self._usage[key]['day_tok'].append((now, tokens))

    def mark_exhausted(self, key):
        with self._lock:
            self._usage[key]['min_req'].extend([time.time()] * self.BUFFERS['rpm'])
            self._idx = (self._idx + 1) % len(self.keys)

    def status(self):
        with self._lock:
            for i, key in enumerate(self.keys):
                self._clean(key); u = self._usage[key]
                print(f"  Key {i+1:2d}: {len(u['min_req']):3d}/{self.BUFFERS['rpm']} RPM  "
                      f"{len(u['day_req']):4d}/{self.BUFFERS['rpd']} RPD")

key_manager = APIKeyManager(API_KEYS_CSV)

# Auto-scale MAX_WORKERS to match the number of loaded API keys
MAX_WORKERS = len(key_manager.keys)
print(f"Workers  : {MAX_WORKERS} (auto-scaled to match {MAX_WORKERS} API keys)")

## Step 4 — Load HotPotQA Data

In [ ]:
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
if N_SAMPLES is not None:
    random.seed(SEED)
    samples = ds.select(random.sample(range(len(ds)), min(N_SAMPLES, len(ds))))
else:
    samples = ds
print(f"Loaded {len(samples)} samples")

## Step 5 — Data Processing

In [ ]:
def process_context(context):
    candidates = []
    for title, sentences in zip(context['title'], context['sentences']):
        for i, sent in enumerate(sentences):
            candidates.append({'title': title, 'sent_id': i, 'text': sent})
    return candidates

def format_gold_supporting_facts(sf):
    return [{'title': t, 'sent_id': s} for t, s in zip(sf['title'], sf['sent_id'])]

## Step 6 — BM25 + Dense Retrievers

In [ ]:
embed_model = SentenceTransformer(DENSE_MODEL)

_lance_db = lancedb.connect("/tmp/lancedb_hotpotqa")
_lance_lock = threading.Lock()

def bm25_retrieve(query, candidates, k=5):
    if not candidates:
        return []
    tokenized = [c['text'].lower().split() for c in candidates]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(query.lower().split())
    top_idx = scores.argsort()[-k:][::-1]
    return [dict(**candidates[i], score=float(scores[i])) for i in top_idx]

def dense_retrieve(query, candidates, k=5):
    if not candidates:
        return []
    texts  = ["passage: " + c['text'] for c in candidates]
    cand_emb = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    q_emb    = embed_model.encode(["query: " + query], convert_to_numpy=True, show_progress_bar=False)[0]
    norms    = np.linalg.norm(cand_emb, axis=1, keepdims=True)
    cand_emb = cand_emb / np.maximum(norms, 1e-10)
    q_norm   = np.linalg.norm(q_emb)
    q_emb    = q_emb / max(q_norm, 1e-10)

    tbl_name = f"tmp_{threading.get_ident()}"
    data = [{"vector": cand_emb[i].tolist(), "idx": i} for i in range(len(candidates))]
    with _lance_lock:
        tbl = _lance_db.create_table(tbl_name, data=data, mode="overwrite")
        results = tbl.search(q_emb.tolist()).limit(min(k, len(candidates))).to_list()
        _lance_db.drop_table(tbl_name)
    return [dict(**candidates[r['idx']], score=float(1.0 / (1.0 + r['_distance'])))
            for r in results]

print(f"Retrievers ready: BM25 + Dense ({DENSE_MODEL})")

## Step 7 — Graph Retriever (Entity Graph + Personalized PageRank)

For each query, builds an in-memory entity graph:
- **Sentence nodes** (S:i) ↔ **Title nodes** (T:title) — weight 1.0
- **Sentence nodes** ↔ **Entity nodes** (E:entity) — weight 1.0
- **Cross-paragraph title mentions** — weight 2.0 (the bridge signal!)

Personalized PageRank propagates from question-matching entities/titles to bridge paragraphs.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
_nlp_lock = threading.Lock()
print("spaCy NER loaded.")

def extract_entities(text):
    with _nlp_lock:
        doc = nlp(text)
    return {ent.text.lower().strip() for ent in doc.ents if len(ent.text.strip()) > 1}

def find_title_mentions(text, other_titles):
    text_lower = text.lower()
    return {t for t in other_titles if t.lower() in text_lower}

def graph_retrieve(query, candidates, k=5):
    if not candidates: return []
    all_titles = set(c['title'] for c in candidates)
    sent_ents = {i: extract_entities(c['text']) for i, c in enumerate(candidates)}
    query_ents = extract_entities(query)

    G = nx.Graph()
    for title in all_titles:
        G.add_node(f"T:{title}", ntype='title', title=title)
    for i, c in enumerate(candidates):
        skey = f"S:{i}"
        G.add_node(skey, ntype='sentence', idx=i)
        G.add_edge(skey, f"T:{c['title']}", weight=1.0)
        for ent in sent_ents[i]:
            ekey = f"E:{ent}"
            if not G.has_node(ekey): G.add_node(ekey, ntype='entity')
            G.add_edge(skey, ekey, weight=1.0)
        for mt in find_title_mentions(c['text'], all_titles - {c['title']}):
            G.add_edge(skey, f"T:{mt}", weight=2.0)

    pers = {}
    for node, data in G.nodes(data=True):
        if data.get('ntype') == 'entity' and node[2:] in query_ents:
            pers[node] = 1.0
    ql = query.lower()
    for node, data in G.nodes(data=True):
        if data.get('ntype') == 'title' and data['title'].lower() in ql:
            pers[node] = 2.0
    if not pers:
        qw = set(ql.split())
        for node, data in G.nodes(data=True):
            if data.get('ntype') == 'title':
                ov = len(qw & set(data['title'].lower().split()))
                if ov: pers[node] = float(ov)

    if pers:
        total = sum(pers.values())
        full_p = {n: pers.get(n, 0.0)/total for n in G.nodes()}
        try: pr = nx.pagerank(G, personalization=full_p, alpha=0.85, max_iter=200)
        except: pr = {n: 1.0/G.number_of_nodes() for n in G.nodes()}
    else:
        pr = {n: 1.0/G.number_of_nodes() for n in G.nodes()}

    scores = {i: pr.get(f"S:{i}", 0.0) for i in range(len(candidates))}
    mx = max(scores.values()) if scores else 1.0
    if mx > 0: scores = {i: s/mx for i, s in scores.items()}
    top = sorted(scores, key=scores.get, reverse=True)[:k]
    return [dict(**candidates[i], score=float(scores[i])) for i in top]

print("Graph retriever ready.")

## Step 8 — Graph-Enhanced Hybrid Retriever (3-Way RRF)

In [ ]:
def graph_hybrid_retrieve(query, candidates, k=5, rrf_k=60, gw=1.5):
    if not candidates: return []
    n = len(candidates)
    sparse = bm25_retrieve(query, candidates, k=n)
    dense  = dense_retrieve(query, candidates, k=n)
    graph  = graph_retrieve(query, candidates, k=n)

    def rank_map(results):
        rm = {}
        for rank, r in enumerate(results, 1):
            for i, c in enumerate(candidates):
                if c['title'] == r['title'] and c['sent_id'] == r['sent_id']:
                    rm[i] = rank; break
        return rm

    sr, dr, gr = rank_map(sparse), rank_map(dense), rank_map(graph)
    rrf = {}
    for i in range(n):
        rrf[i] = (1.0/(rrf_k + sr.get(i, n+1))
                  + 1.0/(rrf_k + dr.get(i, n+1))
                  + gw * (1.0/(rrf_k + gr.get(i, n+1))))
    top = sorted(rrf, key=rrf.get, reverse=True)[:k]
    return [dict(**candidates[i], score=float(rrf[i])) for i in top]

## Step 9 — LLM Client (Multi-Key Groq)

In [ ]:
class GroqClient:
    def __init__(self, model_name, km):
        self.model_name = model_name
        self.km = km
        self._clients = {}
        self._clock = threading.Lock()

    def _client_for(self, api_key):
        with self._clock:
            if api_key not in self._clients:
                self._clients[api_key] = Groq(api_key=api_key)
            return self._clients[api_key]

    def generate(self, prompt, system_prompt=None, max_retries=3):
        for attempt in range(max_retries):
            api_key = self.km.get_key()
            client = self._client_for(api_key)
            try:
                msgs = []
                if system_prompt: msgs.append({"role": "system", "content": system_prompt})
                msgs.append({"role": "user", "content": prompt})
                resp = client.chat.completions.create(
                    model=self.model_name, messages=msgs, max_tokens=512, temperature=0.1)
                tokens = resp.usage.total_tokens if resp.usage else 0
                self.km.record(api_key, tokens)
                return resp.choices[0].message.content
            except Exception as e:
                if '429' in str(e) or 'rate_limit' in str(e).lower():
                    self.km.mark_exhausted(api_key); continue
                print(f"  LLM error: {e}"); return ""
        return ""

    @staticmethod
    def parse_json_output(text):
        try: return json.loads(text)
        except: pass
        m = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except: pass
        m = re.search(r"(\{.*\})", text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except: pass
        return {"answer": "JSON_PARSE_ERROR", "supporting_facts": [], "raw_output": text}

    def predict(self, prompt):
        return self.parse_json_output(self.generate(prompt))

llm = GroqClient(GROQ_MODEL, key_manager)
print("LLM client ready (multi-key).")

## Step 10 — Prompt Construction

In [ ]:
def construct_prompt(question, retrieved_sentences):
    context_str = ""
    for i, item in enumerate(retrieved_sentences, 1):
        context_str += (f"[{i}] Title: {item['title']}\n"
                        f"    Sentence ID: {item['sent_id']}\n"
                        f"    Text: {item['text']}\n\n")
    return f"""You are a helpful assistant for Question Answering.
Answer the following question based ONLY on the provided context sentences.
You must also identify which sentences support your answer.

Context:
{context_str}

Question: {question}

Instructions:
1. Provide a short, concise answer.
2. List the supporting facts as title + sent_id pairs.
3. Use EXACT titles and sent_ids from the context.
4. Output valid JSON only.

Format:
{{{{
  "answer": "...",
  "supporting_facts": [{{{{"title": "...", "sent_id": N}}}}, ...]
}}}}"""

## Step 11 — Evaluator

In [ ]:
def normalize_answer(s):
    s = str(s) if s is not None else ""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(s.split())

def answer_f1(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = collections.Counter(pt) & collections.Counter(gt)
    ns = sum(common.values())
    if ns == 0: return 0.0
    p, r = ns / len(pt), ns / len(gt)
    return (2 * p * r) / (p + r)

def answer_em(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def sp_metrics(pred_sp, gold_sp):
    def to_set(lst):
        return {(x['title'], x['sent_id']) if isinstance(x, dict) else (x[0], x[1]) for x in lst}
    ps, gs = to_set(pred_sp), to_set(gold_sp)
    tp = len(ps & gs)
    prec = tp / len(ps) if ps else 0.0
    rec  = tp / len(gs) if gs else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    em   = 1.0 if ps == gs and len(gs) > 0 else 0.0
    return {'sp_em': em, 'sp_f1': f1, 'sp_prec': prec, 'sp_recall': rec}

## Step 12 — Run GraphRAG (Parallel)

In [ ]:
def process_sample(sample):
    t0 = time.time()
    question   = sample['question']
    candidates = process_context(sample['context'])
    retrieved  = graph_hybrid_retrieve(question, candidates, k=TOP_K, rrf_k=RRF_K, gw=GRAPH_WEIGHT)
    prompt     = construct_prompt(question, retrieved)
    resp       = llm.predict(prompt)
    elapsed    = time.time() - t0
    gold_sp    = format_gold_supporting_facts(sample['supporting_facts'])
    return {
        'pred': {'answer': resp.get('answer', ''), 'supporting_facts': resp.get('supporting_facts', [])},
        'gold': {'answer': sample['answer'], 'supporting_facts': gold_sp},
        'detail': {
            'id': sample['id'], 'question': question,
            'gold_answer': sample['answer'], 'gold_sp': gold_sp,
            'pred_answer': resp.get('answer', ''), 'pred_sp': resp.get('supporting_facts', []),
            'retrieved_context': retrieved, 'time_taken': elapsed,
            'pipeline': PIPELINE_NAME, 'raw_prediction': resp,
        }
    }

experiment_start = datetime.now()
print(f"Starting {PIPELINE_NAME} at {experiment_start.strftime('%H:%M:%S')}")
print(f"{len(samples)} samples × {MAX_WORKERS} workers\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    fmap = {executor.submit(process_sample, s): i for i, s in enumerate(samples)}
    for future in tqdm(as_completed(fmap), total=len(fmap), desc=PIPELINE_NAME):
        try:
            r = future.result(); r['idx'] = fmap[future]; results.append(r)
        except Exception as e:
            print(f"  Error on sample {fmap[future]}: {e}")

results.sort(key=lambda x: x['idx'])
predictions = [r['pred']   for r in results]
golds       = [r['gold']   for r in results]
details     = [r['detail'] for r in results]

experiment_end = datetime.now()
total_time = (experiment_end - experiment_start).total_seconds()
print(f"\nDone in {total_time:.1f}s ({total_time/len(samples):.2f}s/sample)")

## Step 13 — Evaluation

In [ ]:
metrics = {'em':0,'f1':0,'sp_em':0,'sp_f1':0,'sp_prec':0,'sp_recall':0,'joint_em':0,'joint_f1':0}
for pred, gold in zip(predictions, golds):
    em = answer_em(pred['answer'], gold['answer'])
    f1 = answer_f1(pred['answer'], gold['answer'])
    sp = sp_metrics(pred['supporting_facts'], gold['supporting_facts'])
    metrics['em'] += em; metrics['f1'] += f1
    metrics['sp_em'] += sp['sp_em']; metrics['sp_f1'] += sp['sp_f1']
    metrics['sp_prec'] += sp['sp_prec']; metrics['sp_recall'] += sp['sp_recall']
    metrics['joint_em'] += em * sp['sp_em']; metrics['joint_f1'] += f1 * sp['sp_f1']
n = len(predictions)
for k in metrics: metrics[k] /= n

print(f"{'='*50}")
print(f"  {PIPELINE_NAME} Results ({n} samples)")
print(f"{'='*50}")
for k, v in metrics.items(): print(f"  {k:20s}: {v:.4f}")
print(f"{'='*50}")

## RAGAS Faithfulness Evaluation

Measures what fraction of the generated answer's claims can be inferred from the retrieved context.

In [ ]:
# --- RAGAS Faithfulness (parallel-wave, all keys) ---
print("\nRunning RAGAS Faithfulness evaluation (parallel waves)...")

import warnings, time
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore', category=DeprecationWarning, module='ragas')

from ragas import evaluate as ragas_evaluate
from ragas.metrics import Faithfulness
from ragas.run_config import RunConfig
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

faithfulness_metric = Faithfulness()

# --- tunables ----------------------------------------------------------
WAVE_COOLDOWN  = 15   # seconds between waves (let TPM window breathe)
SAMPLE_TIMEOUT = 180  # per-sample RAGAS timeout
MAX_RETRIES    = 3    # retries per sample on 429
# -----------------------------------------------------------------------

ragas_run_cfg = RunConfig(timeout=SAMPLE_TIMEOUT, max_retries=3, max_wait=30)

# Build per-sample dicts once
sample_dicts = []
for d in details:
    ctx = [item["text"] for item in d.get("retrieved_context", [])
           if isinstance(item, dict) and "text" in item]
    sample_dicts.append({
        "question": d.get("question", ""),
        "answer":   d.get("pred_answer", ""),
        "contexts": ctx if ctx else [""],
    })

n_keys    = len(key_manager.keys)
n_samples = len(sample_dicts)
print(f"{n_samples} samples | {n_keys} keys | ~{(n_samples + n_keys - 1) // n_keys} waves")


def evaluate_one(sample_idx, api_key, key_no):
    """Evaluate a single sample with the given API key. Returns (idx, score)."""
    ds = Dataset.from_dict({
        "question": [sample_dicts[sample_idx]["question"]],
        "answer":   [sample_dicts[sample_idx]["answer"]],
        "contexts": [sample_dicts[sample_idx]["contexts"]],
    })
    for attempt in range(MAX_RETRIES):
        try:
            llm = ChatGroq(
                model="llama-3.3-70b-versatile",
                groq_api_key=api_key,
                temperature=0,
            )
            res = ragas_evaluate(
                ds,
                metrics=[Faithfulness()],
                llm=LangchainLLMWrapper(llm),
                run_config=ragas_run_cfg,
            )
            score = res.to_pandas()["faithfulness"].iloc[0]
            return (sample_idx, score)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower():
                key_manager.mark_exhausted(api_key)
                api_key = key_manager.get_key()
                key_no  = key_manager.keys.index(api_key) + 1
                wait = 5 * (attempt + 1)
                print(f"    [sample {sample_idx+1}] 429 → rotated to key #{key_no}, "
                      f"retry {attempt+1}/{MAX_RETRIES} after {wait}s")
                time.sleep(wait)
            else:
                print(f"    [sample {sample_idx+1}] error: {e}")
                return (sample_idx, None)
    return (sample_idx, None)


# --- Run in waves of n_keys -------------------------------------------
scores = [None] * n_samples
wave = 0
i = 0  # next sample index to schedule

while i < n_samples:
    wave += 1
    wave_end = min(i + n_keys, n_samples)
    wave_size = wave_end - i
    print(f"\n  Wave {wave}: samples {i+1}-{wave_end} ({wave_size} in parallel)")

    futures = {}
    with ThreadPoolExecutor(max_workers=wave_size) as pool:
        for j in range(wave_size):
            sample_idx = i + j
            key_idx    = j % n_keys
            api_key    = key_manager.keys[key_idx]
            key_no     = key_idx + 1
            fut = pool.submit(evaluate_one, sample_idx, api_key, key_no)
            futures[fut] = (sample_idx, key_no)

        for fut in as_completed(futures):
            sidx, kno = futures[fut]
            idx, score = fut.result()
            tag = f"{score:.4f}" if score is not None and score == score else "FAIL"
            print(f"    sample {idx+1:2d}  key #{kno:2d}  → {tag}")
            scores[idx] = score

    i = wave_end
    if i < n_samples:
        print(f"  Cooling down {WAVE_COOLDOWN}s before next wave...")
        time.sleep(WAVE_COOLDOWN)

# --- Aggregate ---------------------------------------------------------
valid = [s for s in scores if s is not None and s == s]
avg_faithfulness = sum(valid) / len(valid) if valid else 0.0
metrics['faithfulness'] = avg_faithfulness

print(f"\n{'='*50}")
print(f"  RAGAS Faithfulness: {avg_faithfulness:.4f}")
print(f"  (computed on {len(valid)}/{n_samples} samples)")
print(f"{'='*50}")

## Step 14 — Save Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
ts = experiment_start.strftime('%Y%m%d_%H%M%S')
out_file = f'{RESULTS_DIR}/graph_{ts}_results.json'
with open(out_file, 'w') as f:
    json.dump({'args': {'pipeline': PIPELINE_NAME, 'model': GROQ_MODEL, 'dense_model': DENSE_MODEL,
               'top_k': TOP_K, 'rrf_k': RRF_K, 'graph_weight': GRAPH_WEIGHT,
               'n_samples': N_SAMPLES, 'max_workers': MAX_WORKERS},
               'metrics': metrics, 'timing': {'total_s': total_time, 'avg_s': total_time/n},
               'details': details,
               'faithfulness_per_sample': globals().get('faithfulness_scores', [])}, f, indent=2)
print(f"Saved → {out_file}")

## Step — Export Metrics CSV


In [ ]:
# --- Export metrics to CSV ---------------------------------------------------
import csv, os

CSV_PATH = f"{BASE_DIR}/results/experiment_metrics_e5.csv"
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# Derive a unique pipeline label  (e.g. "VanillaRAG_FAISS", "GraphRAG_Qdrant")
_store = RESULTS_DIR.rstrip("/").split("/")[-1]            # faiss / lancedb / qdrant
_label = PIPELINE_NAME
if _store.lower() not in _label.lower():                   # VanillaRAG has no suffix
    _label = f"{PIPELINE_NAME}_{_store.upper()}"

row = {
    "Pipeline":           _label,
    "em":                 round(metrics.get("em", 0), 4),
    "f1":                 round(metrics.get("f1", 0), 4),
    "sp_em":              round(metrics.get("sp_em", 0), 4),
    "sp_f1":              round(metrics.get("sp_f1", 0), 4),
    "sp_prec":            round(metrics.get("sp_prec", 0), 4),
    "sp_recall":          round(metrics.get("sp_recall", 0), 4),
    "joint_em":           round(metrics.get("joint_em", 0), 4),
    "joint_f1":           round(metrics.get("joint_f1", 0), 4),
    "RAGAS Faithfulness": round(metrics.get("faithfulness", 0), 4),
}

header = list(row.keys())
write_header = not os.path.exists(CSV_PATH)

with open(CSV_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=header)
    if write_header:
        w.writeheader()
    w.writerow(row)

print(f"Appended metrics row to {CSV_PATH}")
print("  ", row)

## Step 15 — Inspect Predictions & Key Usage

In [ ]:
for d in details[:5]:
    print(f"Q: {d['question']}")
    print(f"  Gold: {d['gold_answer']} | Pred: {d['pred_answer']}")
    print(f"  Retrieved: {[(r['title'], r['sent_id']) for r in d['retrieved_context']]}\n")

print("--- API Key Usage ---")
key_manager.status()